# ML Laboratuvarı · Modül 7, Ünite 9: Ülke kümeleri ve FOMC tonu

**Prof. Dr. Mehmet Songur** · ML Laboratuvarı, Modül 7

Veri: Dünya Bankası Dünya Kalkınma Göstergeleri, 157 ülkenin 2019–2023 ortalamaları (CC BY 4.0); 2002'den bu yana FOMC para politikası açıklamaları ve aylık federal fon oranı (FRED; ABD kamu malı). Kurulum hücresi `ulke`, `bildiri` ve `faiz`'i indirir; `deg`, `sahin`, `guvercin` listeleri ve `degisim(t, m)` fonksiyonu hazırdır. Kodu **Gemini'ye yazdır**, sonra **denetle**.

**Önce kopyala:** **Dosya → Drive'a kopya kaydet** ile kendi kopyanda çalış. Hücrelere başlıklarıyla başvuruyoruz: Kurulum, Görev A, Kontrol A, Görev B, Kontrol B.

**Kurallar**
1. **Kurulum** ve **Kontrol** hücrelerini değiştirme; yalnızca çalıştır.
2. Görev hücrelerinde kodu Gemini ile üret; çalıştırmadan önce oku.
3. Kontrol hücresi geçerse bir **tamamlama kodu** verir; onu ML Laboratuvarı'ndaki kutuya yapıştır.

In [ ]:
#@title Kurulum — çalıştır, değiştirme

import base64, re, warnings
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score
warnings.filterwarnings("ignore")

def _k(s):
    return base64.b64decode(s).decode()

_D = ["gdppc_ppp", "inflation", "unemployment", "trade", "agriculture", "urban", "life_exp", "fertility"]
_S = ["tighten", "tightening", "firming", "raise", "increase", "elevated", "pressures", "upside", "strong", "robust", "solid", "reduce"]
_G = ["accommodative", "accommodation", "ease", "easing", "lower", "decline", "weak", "weakened", "slack", "downside", "subdued", "purchases"]

def _kume(X):
    Z = StandardScaler().fit_transform(X)
    sil = {k: silhouette_score(Z, KMeans(n_clusters=k, n_init=10, random_state=0).fit_predict(Z)) for k in range(2, 9)}
    k = max(sil, key=sil.get)
    return Z, k, sil[k], KMeans(n_clusters=k, n_init=10, random_state=0).fit_predict(Z)

def _referans(u, b, f):
    X = u[_D].copy()
    X["gdppc_ppp"] = np.log(X["gdppc_ppp"]); X["inflation"] = np.log(X["inflation"])
    Z, k, s, e = _kume(X)
    out = dict(Z=Z, k=k, sil=s, ari=adjusted_rand_score(u["income"], e))
    e3 = KMeans(n_clusters=3, n_init=10, random_state=0).fit_predict(Z)
    out.update(sil3=silhouette_score(Z, e3), ari3=adjusted_rand_score(u["income"], e3), boy=np.bincount(e).tolist())
    ine = {k: KMeans(n_clusters=k, n_init=10, random_state=0).fit(Z).inertia_ for k in range(2, 9)}
    out["k_inertia"] = min(ine, key=ine.get)
    def say(m, ks, sinir=True):
        return sum(len(re.findall(r"\b" + w + r"\b", m)) if sinir else m.count(w) for w in ks)
    def ton(metinler, sinir=True):
        s = metinler.map(lambda m: say(m, _S, sinir)); g = metinler.map(lambda m: say(m, _G, sinir))
        return (s - g) / (s + g).clip(lower=1)
    def deg(t, m):
        son = (t + pd.DateOffset(months=m)).replace(day=1)
        return f.asof(son) - f.asof(t.replace(day=1)) if son <= f.index[-1] else np.nan
    once = b["date"].map(lambda t: f.asof(t.replace(day=1)) - f.asof((t - pd.DateOffset(months=3)).replace(day=1)))
    d3 = b["date"].map(lambda t: deg(t, 3))
    out["ton"] = ton(b["text"].str.lower())
    low = b["text"].str.lower()
    out["bos"] = int(((low.map(lambda m: say(m, _S)) + low.map(lambda m: say(m, _G))) == 0).sum())
    out["ton_sub"] = ton(b["text"].str.lower(), False)
    out["ton_case"] = ton(b["text"])
    out["d3"] = d3
    out["kor"] = out["ton"].corr(d3)
    out["kor_once"] = out["ton"].corr(once)
    return out

ulke = pd.read_csv("https://raw.githubusercontent.com/mehmetsongur/ml-lab/main/data/wb/countries.csv")
bildiri = pd.read_csv("https://raw.githubusercontent.com/mehmetsongur/ml-lab/main/data/fomc/statements.csv", parse_dates=["date"])
faiz = pd.read_csv("https://raw.githubusercontent.com/mehmetsongur/ml-lab/main/data/fred/FEDFUNDS.csv", index_col=0, parse_dates=True).iloc[:, 0]
deg = list(_D)
sahin, guvercin = list(_S), list(_G)

def degisim(t, m):
    """Açıklama ayından m ay sonraki ayın ortalama federal fon oranı eksi açıklama ayının ortalaması (puan).
    FEDFUNDS aylık ortalamadır: bu, hedef aralıktaki bir kararın yönü değil, iki aylık ortalamanın farkıdır."""
    son = (t + pd.DateOffset(months=m)).replace(day=1)
    return faiz.asof(son) - faiz.asof(t.replace(day=1)) if son <= faiz.index[-1] else np.nan

print(f"ulke: {len(ulke)} ülke · bildiri: {len(bildiri)} açıklama ({bildiri['date'].min():%Y-%m} – {bildiri['date'].max():%Y-%m}) · faiz: {faiz.index[-1]:%Y-%m}'e kadar")
_R = _referans(ulke, bildiri, faiz)

def kontrol_A(Z, k_en_iyi, siluet, ari):
    Z = np.asarray(Z)
    if Z.shape != _R["Z"].shape:
        print(f"❌ Z'nin boyutu {Z.shape}, beklenen {_R['Z'].shape}: sekiz göstergenin hepsi, 157 ülke."); return
    if not np.allclose(Z, _R["Z"]):
        if np.allclose(Z, StandardScaler().fit_transform(ulke[deg])):
            print("❌ Z'de logaritma yok: önce gdppc_ppp ve inflation için np.log, sonra StandardScaler."); return
        print("❌ Z beklenenden farklı: X'te gdppc_ppp ve inflation'ın np.log'u alınmalı, sonra StandardScaler().fit_transform(X)."); return
    if int(k_en_iyi) != _R["k"]:
        if int(k_en_iyi) == _R["k_inertia"]:
            print("❌ k inertia ile seçilmiş: en iyi ulaşılabilir inertia k arttıkça artmaz, bu yüzden hep aralığın sonunu seçer. Siluet skoru en yüksek k'yı seç."); return
        print("❌ k_en_iyi beklenenden farklı: k = 2…8, KMeans(n_clusters=k, n_init=10, random_state=0), silhouette_score en yüksek k."); return
    if abs(float(siluet) - _R["sil"]) > 1e-6:
        print("❌ siluet beklenenden farklı: seçilen k için silhouette_score(Z, etiketler)."); return
    if abs(float(ari) - _R["ari"]) > 1e-6:
        print("❌ ari beklenenden farklı: seçilen k'nın etiketleriyle adjusted_rand_score(ulke['income'], etiketler)."); return
    print(f"Siluete göre seçilen k = {_R['k']} (küme büyüklükleri {_R['boy']}): siluet {_R['sil']:.3f}, ARI {_R['ari']:.3f}. Karşılaştırma: k = 3 için siluet {_R['sil3']:.3f}, ARI {_R['ari3']:.3f}. “En iyi k” seçilen ölçüte bağlıdır; denetim notunda iki çözümü tartış.")
    print(f"✅ Tüm kontroller geçti (siluete göre k = {_R['k']}, siluet {_R['sil']:.3f}, ARI {_R['ari']:.3f}). Tamamlama kodun: " + _k("TUw3LUtVTS01UldE"))

def kontrol_B(ton, korelasyon):
    ton = pd.Series(np.asarray(ton, dtype=float))
    if len(ton) != len(bildiri):
        print(f"❌ ton {len(ton)} değer içeriyor; her açıklama için bir değer olmalı ({len(bildiri)})."); return
    if not np.allclose(ton.values, _R["ton"].values, equal_nan=True):
        if np.allclose(ton.values, _R["ton_sub"].values, equal_nan=True):
            print("❌ Kelimeler str.count ile sayılmış: 'increase' içindeki 'ease' de sayılıyor. re.findall(r'\\b' + k + r'\\b', metin) kullan."); return
        if np.allclose(ton.values, _R["ton_case"].values, equal_nan=True):
            print("❌ Metinler küçük harfe çevrilmemiş: str.lower()."); return
        print("❌ ton beklenenden farklı: (şahin − güvercin) / (şahin + güvercin), kelime yoksa 0; küçük harf ve kelime sınırları."); return
    if abs(float(korelasyon) - _R["kor"]) > 1e-6:
        if abs(float(korelasyon) - _R["kor_once"]) < 1e-6:
            print("❌ Faiz değişimi açıklamadan ÖNCEKİ 3 ayla hesaplanmış: sonraki 3 ay, degisim(t, 3)."); return
        print("❌ korelasyon beklenenden farklı: ton ile bildiri['date'].map(lambda t: degisim(t, 3)) arasındaki korelasyon (pandas corr)."); return
    print(f"Toplam {len(bildiri)} açıklama; 3 aylık faiz farkı hesaplanabilen {int(_R['d3'].notna().sum())}; hiç sözlük kelimesi olmayan (ton = 0, payda sıfır kuralı) {_R['bos']}.")
    print(f"✅ Tüm kontroller geçti (ortalama ton {_R['ton'].mean():.3f}, 3 aylık korelasyon {_R['kor']:.3f}). Tamamlama kodun: " + _k("TUw3LVRPTi04R1BM"))

ulke.head(3)

## Görev A — Ülke kümeleri

1. `X = ulke[deg].copy()`; `X`'te `gdppc_ppp` ve `inflation` sütunlarının `np.log`'unu al.
2. `Z = StandardScaler().fit_transform(X)`.
3. k = 2'den 8'e kadar `KMeans(n_clusters=k, n_init=10, random_state=0)` kur ve `silhouette_score(Z, etiketler)`'i hesapla. Skoru en yüksek k: `k_en_iyi`; onun skoru: `siluet`.
4. Seçilen k'nın etiketleriyle `adjusted_rand_score(ulke["income"], etiketler)`: `ari`.
5. Seçilen çözümün küme büyüklüklerine ve her kümedeki birkaç ülkeye bak.

**Denetim soruları:** Logaritma ölçeklemeden önce mi alındı? k inertia ile mi seçildi? Küme büyüklükleri makul mü?

In [ ]:
# Görev A: Gemini ile kodu burada üret, oku, sonra çalıştır.

### Kontrol A — çalıştır, değiştirme

In [ ]:
kontrol_A(Z, k_en_iyi, siluet, ari)

## Görev B — Ton endeksi

1. Metinleri küçük harfe çevir: `bildiri["text"].str.lower()`.
2. Her açıklamada `sahin` ve `guvercin` listelerindeki kelimeleri **kelime sınırlarıyla** say: `len(re.findall(r"\b" + k + r"\b", metin))`.
3. `ton = (şahin − güvercin) / (şahin + güvercin)`; iki sayı da sıfırsa ton 0 (`(s + g).clip(lower=1)` ile böl).
4. Sonraki 3 ayda faiz değişimi: `bildiri["date"].map(lambda t: degisim(t, 3))`. `ton` ile korelasyonu (pandas `corr`): `korelasyon`.
5. Tonun yıllık ortalamasını faizle yan yana koy.

**Denetim soruları:** `str.count` var mı? Değişim açıklamadan sonrasına mı bakıyor? Bu korelasyon bir öngörü kanıtı mı?

In [ ]:
# Görev B: Gemini ile kodu burada üret, oku, sonra çalıştır.

### Kontrol B — çalıştır, değiştirme

In [ ]:
kontrol_B(ton, korelasyon)

## Denetim notların

- Gemini'nin kümeleme kodunda neyi düzeltmen gerekti?
- Siluete göre seçilen k ile k = 3'ü karşılaştır: k = 3'ün siluet skoru düşük ama gelir gruplarıyla uyumu (ARI) yüksek. Analizin iktisadi amacı değişirse hangi ölçüt önem kazanır?
- Seçtiğin küme çözümünü bir politika raporunda nasıl anlatırsın? (k, siluet, ARI, küme üyeleri, sınırlılıklar)
- Ton endeksi faizle ilişkili mi? Bu ilişki öngörü mü, yoksa kararın duyurulması mı?